# LangGraph Module · Day 03 — State & Memory

**Phase 1 · Module 1: LangGraph & LangChain Deep Dive** · ~2.5 hrs

### Today's tasks (from the plan)
- [ ] Study **`TypedDict` state schema design** (what to put in state, and how updates merge)
- [ ] Add **persistent memory** using **`SqliteSaver`** & **`MemorySaver`**
- [ ] Build a **3-turn conversation loop** with state preserved across turns

🐍 **Python (30 min):** *Async/Await Fundamentals* — see `Python/Day03.ipynb`. Today's LLM calls are the exact thing you'd `await` in production.

### The one big idea
Day 2 gave the graph a checkpointer (`MemorySaver`) so state survived **across `invoke` calls within one process**. But `MemorySaver` lives in RAM — restart the kernel and every conversation is gone. Today we swap in **`SqliteSaver`**: the **same checkpointer interface**, but state is written to a **file on disk**, so a conversation survives a restart. That is the difference between a demo and an agent a user can come back to.

## 0. Setup — API key + the SQLite checkpointer package

Runs against **Google Gemini** (needs `GOOGLE_API_KEY` in a `.env` at the project root). The SQLite saver ships in a **separate** package from core `langgraph`:

```bash
uv pip install langgraph langchain-google-genai python-dotenv langgraph-checkpoint-sqlite
```

The cell below installs `langgraph-checkpoint-sqlite` only if it's missing, so the notebook is self-contained.

In [1]:
# Install the SQLite checkpointer only if it isn't already present
try:
    from langgraph.checkpoint.sqlite import SqliteSaver
    print("langgraph-checkpoint-sqlite already installed ✅")
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "langgraph-checkpoint-sqlite"])
    print("installed langgraph-checkpoint-sqlite ✅ (restart not needed)")

langgraph-checkpoint-sqlite already installed ✅


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
HAS_KEY = bool(os.getenv("GOOGLE_API_KEY"))
print("GOOGLE_API_KEY:", "loaded ✅" if HAS_KEY else "MISSING ❌ — will use a stub LLM")

GOOGLE_API_KEY: loaded ✅


## 1. State schema design — what belongs in state?

State is the graph's **shared memory for one run/thread**. Good state design follows three questions per field:

| Question | If yes → | Example |
|----------|----------|---------|
| Does more than one node read/write it? | put it in state | `messages` |
| Should updates **accumulate**? | give it a **reducer** (`Annotated[..., add_messages]`) | chat history |
| Is it set once / overwritten each turn? | plain field (default overwrite) | `turn`, `user_name` |

For a chat agent the workhorse field is `messages` with the **`add_messages`** reducer, so every turn **appends** to history instead of clobbering it (Day 2's lesson, now put to work).

In [3]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]   # APPEND-reducer: full history kept
    turn: int                                 # plain field: overwritten each turn

print("ChatState fields:", list(ChatState.__annotations__))

ChatState fields: ['messages', 'turn']


## 2. The chat node — one LLM turn

The node reads the **whole `messages` history**, sends it to the model, and returns the reply as a **partial update**. Because `messages` has the `add_messages` reducer, returning `[reply]` **appends** — history grows every turn.

We define a tiny `call_llm` that uses the real Gemini model if a key is present, else a deterministic **stub** so the notebook always runs.

In [4]:
from langchain_core.messages import AIMessage, HumanMessage

if HAS_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    _model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    def call_llm(messages: list):
        return _model.invoke(messages)                     # -> AIMessage
else:
    def call_llm(messages: list):
        # Stub: echoes the latest human turn + how many turns it has seen,
        # which is enough to PROVE memory is working.
        last = next((m.content for m in reversed(messages)
                     if isinstance(m, HumanMessage)), "")
        seen = sum(isinstance(m, HumanMessage) for m in messages)
        return AIMessage(content=f"(stub) you said {last!r}; I've seen {seen} of your turns")

def chat_node(state: ChatState) -> dict:
    reply = call_llm(state["messages"])          # sees ALL prior turns
    return {"messages": [reply], "turn": state.get("turn", 0) + 1}

print("model ready:", "Gemini" if HAS_KEY else "stub")

model ready: Gemini


## 3. Build the graph

Minimal shape: `START → chat → END`. The interesting part isn't the topology — it's the **checkpointer** we attach at `compile()`, which is what gives the graph memory across calls.

In [5]:
def build_graph(checkpointer):
    b = StateGraph(ChatState)
    b.add_node("chat", chat_node)
    b.add_edge(START, "chat")
    b.add_edge("chat", END)
    return b.compile(checkpointer=checkpointer)   # <- memory lives here

print("builder ready")

builder ready


## 4. `MemorySaver` — memory in RAM (recap from Day 2)

`MemorySaver` stores checkpoints in a dict in memory. State persists **across `invoke` calls in this process**, keyed by `thread_id` — but it's **gone on kernel restart**. Good for tests and demos.

In [6]:
from langgraph.checkpoint.memory import MemorySaver

mem_graph = build_graph(MemorySaver())
cfg = {"configurable": {"thread_id": "demo-1"}}

# Only pass the NEW human message each turn -- the reducer appends it to history
mem_graph.invoke({"messages": [HumanMessage("Hi, my name is Gaurav.")]}, cfg)
out = mem_graph.invoke({"messages": [HumanMessage("What did I just tell you?")]}, cfg)

print("turns recorded:", out["turn"])                 # 2 -> state carried over
print("history length:", len(out["messages"]))        # 4 msgs (2 human + 2 ai)
print("last reply    :", out["messages"][-1].content)

turns recorded: 2
history length: 4
last reply    : You just told me your name is Gaurav.


☝️ Second `invoke` sent **only** the new question, yet `turn == 2` and history has 4 messages — the checkpointer reloaded the prior state for `thread_id="demo-1"` and the reducer appended. That's memory. But restart the kernel and it vanishes. Enter SQLite.

## 5. `SqliteSaver` — persistent memory on disk

**Same interface**, durable storage. `SqliteSaver` writes each checkpoint to a SQLite **file**, so state survives a process/kernel restart. Swapping `MemorySaver()` → `SqliteSaver(...)` is the *only* change needed — the graph code is identical.

`SqliteSaver.from_conn_string(path)` is a **context manager**; keep the connection open while you use the graph.

In [7]:
import os, sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH = "day03_memory.sqlite"
if os.path.exists(DB_PATH):     # start clean so re-runs are reproducible
    os.remove(DB_PATH)

# Open a raw sqlite connection ourselves so it stays alive across cells.
# (check_same_thread=False lets the notebook's threads share it.)
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
sqlite_saver = SqliteSaver(conn)

sql_graph = build_graph(sqlite_saver)
cfg = {"configurable": {"thread_id": "gaurav-session"}}
print("SqliteSaver wired to", DB_PATH, "✅")

SqliteSaver wired to day03_memory.sqlite ✅


## 6. The deliverable — a 3-turn conversation loop

Feed three user turns one at a time. Each `invoke` passes **only the new message**; the checkpointer reloads history and the reducer appends. Watch `turn` climb 1 → 2 → 3 and the reply reference earlier turns.

In [8]:
user_turns = [
    "Hi, my name is Gaurav and I'm learning LangGraph.",
    "What is my name?",
    "Remind me — what am I learning?",
]

for i, text in enumerate(user_turns, start=1):
    out = sql_graph.invoke({"messages": [HumanMessage(text)]}, cfg)
    print(f"--- turn {i} (state turn={out['turn']}) ---")
    print("user     :", text)
    print("assistant:", out["messages"][-1].content)
    print()

print("total messages in history:", len(out["messages"]))   # 6 = 3 human + 3 ai

--- turn 1 (state turn=1) ---
user     : Hi, my name is Gaurav and I'm learning LangGraph.
assistant: Hi Gaurav! That's fantastic! LangGraph is a really powerful and exciting framework for building robust, stateful LLM applications.

I'd be happy to help you on your learning journey. What are you currently working on, or what specific questions do you have about LangGraph?

For example, are you interested in:
*   Understanding the core concepts (nodes, edges, state)?
*   Building your first agent?
*   Creating multi-agent systems?
*   Debugging a specific issue?
*   Exploring advanced features like human-in-the-loop or tool calling?

Let me know how I can assist!

--- turn 2 (state turn=2) ---
user     : What is my name?
assistant: Your name is Gaurav!

--- turn 3 (state turn=3) ---
user     : Remind me — what am I learning?
assistant: You are learning **LangGraph**!

total messages in history: 6


## 7. Prove it's really on disk — reload from a fresh graph

The real test of *persistence*: build a **brand-new graph object** from a **new connection to the same file**, and read the state back **without sending any message**. If the 3 turns come back, they were genuinely on disk — this is what would survive a kernel restart.

In [9]:
# Simulate a restart: throw away sql_graph/conn, open the file afresh.
conn2 = sqlite3.connect(DB_PATH, check_same_thread=False)
reloaded = build_graph(SqliteSaver(conn2))

# get_state reads the LATEST checkpoint for this thread -- no invoke needed
snapshot = reloaded.get_state(cfg)
print("recovered turn count:", snapshot.values["turn"])            # -> 3
print("recovered history len:", len(snapshot.values["messages"]))  # -> 6
print("first thing user said:", snapshot.values["messages"][0].content)

recovered turn count: 3
recovered history len: 6
first thing user said: Hi, my name is Gaurav and I'm learning LangGraph.


☝️ A fresh graph, a fresh connection, **no new message** — and the full 3-turn conversation is right there. That is durable memory. Continuing the thread just picks up at turn 4:

In [10]:
out = reloaded.invoke({"messages": [HumanMessage("One more — still remembered?")]}, cfg)
print("turn now:", out["turn"])                       # -> 4, continued seamlessly
print("reply   :", out["messages"][-1].content)

# A DIFFERENT thread_id is isolated -- separate conversation, separate row
other = reloaded.invoke({"messages": [HumanMessage("who am I?")]},
                        {"configurable": {"thread_id": "someone-else"}})
print("other thread turn:", other["turn"])            # -> 1, fresh

turn now: 4
reply   : Yes, I still remember! You are learning **LangGraph**.
other thread turn: 1


In [11]:
# Tidy up the open connections (and optionally the file)
for c in (conn, conn2):
    try:
        c.close()
    except Exception:
        pass
print("connections closed. DB file left on disk:", os.path.exists(DB_PATH))

connections closed. DB file left on disk: True


## 8. `MemorySaver` vs `SqliteSaver` — when to use which

| | **`MemorySaver`** | **`SqliteSaver`** |
|--|--|--|
| Stores state in | a dict in RAM | a SQLite **file** on disk |
| Survives kernel/process restart | ❌ no | ✅ yes |
| Setup | zero | one file path (or `:memory:`) |
| Interface | identical `.invoke/.get_state` | identical `.invoke/.get_state` |
| Use for | tests, quick demos | local dev, single-node apps, notebooks you revisit |
| Production at scale | — | swap to `PostgresSaver` (same interface again) |

**Takeaway:** the checkpointer is a **pluggable backend**. Your graph code never changes — only the storage does. `MemorySaver` → `SqliteSaver` → `PostgresSaver` is a one-line swap as you go from test → local → production.

## 9. Recap

| Concept | What it is | Why it's used |
|---------|-----------|---------------|
| **state design** | pick fields; `Annotated[..., add_messages]` for history | shared, accumulating memory for a thread |
| **`thread_id`** | key in `config["configurable"]` | separates one conversation from another |
| **`MemorySaver`** | in-RAM checkpointer | fast, disposable memory for demos/tests |
| **`SqliteSaver`** | file-backed checkpointer | **durable** memory that survives restarts |
| **`get_state(cfg)`** | read latest checkpoint without invoking | inspect/resume a saved conversation |
| **partial input** | send only the new message each turn | reducer + checkpointer rebuild full history |
